# Build Simple State Model with AsyncMachineBuilder

## 1. State Diagram

```lua
stateDiagram-v2
    INIT --> GENERATE
    GENERATE --> FINAL
```

```mermaid
stateDiagram-v2
    INIT --> GENERATE
    GENERATE --> FINAL
```

---

## 2. Use AsyncMachineBuilder


In [1]:
from gai.asm import AsyncStateMachine

async def generate_action(state):
    
    from openai import AsyncOpenAI
    client = AsyncOpenAI()
    
    # Import data from state_bag
    agent_name = state.machine.state_bag.get("name", "Assistant")
    user_message = state.machine.state_bag.get("user_message", "If you are seeing this, that means I have forgotten to add a user message. Remind me.")
    monologue_messages = state.input["monologue_messages"]
    
    from openai import AsyncOpenAI
    client = AsyncOpenAI()
    
    # Execute
    
    monologue_messages.append({"role": "system", "content": f"Your name is {agent_name}. You are a helpful assistant."})
    monologue_messages.append({"role": "user", "content": f"{agent_name}, {user_message}"})
    response = await client.chat.completions.create(
        model="gpt-4.1",
        messages=monologue_messages,
        max_tokens=50,
        stream=True
    )
    
    async def streamer():
        content = ""
        async for chunk in response:
            chunk = chunk.choices[0].delta.content
            if isinstance(chunk,str) and chunk:
                content += chunk
                yield chunk
        monologue_messages.append({
            "role": "assistant",
            "content": content
        })

    state.machine.state_bag["streamer"] = streamer()

## Step 1: INIT

with AsyncStateMachine.StateMachineBuilder("""
    INIT --> GENERATE
    GENERATE --> FINAL
    """
    ) as builder:
    fsm = builder.build({
        "INIT": {
            "input_data": {
                "name": "Sara",
                "user_message": "Hello, world!",
                "dialogue_messages": []
            },
        },
        "GENERATE": {
            "module_path": "gai.asm.states",
            "class_name": "PureActionState",
            "title": "GENERATE",
            "action": "generate",
            "output_data": ["streamer"]            
        },
        "FINAL": {
            "output_data": ["monologue_messages"],
        }
    },generate=generate_action)

## Step 2: INIT --> GENERATE

await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk, end='', flush=True)
print("\n\n")

## Step 3: GENERATE --> FINAL
await fsm.run_async()

## Step 4: Print the state history
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)

Hello, world! 😊 How can I help you today?


State History:
State: INIT
- input: {'name': 'Sara', 'user_message': 'Hello, world!', 'dialogue_messages': []}
- output: {'name': 'Sara', 'monologue_messages': [{'role': 'system', 'content': 'Your name is Sara. You are a helpful assistant.'}, {'role': 'user', 'content': 'Sara, Hello, world!'}, {'role': 'assistant', 'content': 'Hello, world! 😊 How can I help you today?'}], 'step': 0, 'time': datetime.datetime(2025, 6, 11, 6, 52, 11, 751299)}
--------------------
State: GENERATE
- input: {'name': 'Sara', 'monologue_messages': [{'role': 'system', 'content': 'Your name is Sara. You are a helpful assistant.'}, {'role': 'user', 'content': 'Sara, Hello, world!'}, {'role': 'assistant', 'content': 'Hello, world! 😊 How can I help you today?'}], 'step': 1, 'time': datetime.datetime(2025, 6, 11, 6, 52, 11, 751557)}
- output: {'streamer': <async_generator object generate_action.<locals>.streamer at 0x781a94edb6c0>, 'name': 'Sara', 'monologue_messages': [{